In [ ]:
import os
import datetime
import numpy as np

import h5py
from tifffile import imread
from skimage import transform

import tensorflow as tf
from tensorflow.python.platform import build_info as tf_build_info
from tensorflow import keras
from tensorflow.keras import layers

2026-04-13 23:15:48.844320: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-13 23:15:48.844419: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-13 23:15:49.251916: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-13 23:15:49.741941: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print("Tensorflow Version: ", tf.__version__)
print("CUDA Build Version:", tf_build_info.build_info['cuda_version'])
print("GPU:", len(tf.config.list_physical_devices('GPU'))>0)

Tensorflow Version:  2.15.0
CUDA Build Version: 12.0
GPU: True


2026-04-13 23:16:08.596835: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-13 23:16:10.666696: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-13 23:16:10.666914: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [ ]:
class UNet(keras.Model):
    def __init__(self, input_size=(None, None, 1)):
        d0a = layers.Input(input_size, name='input')
        d0b = layers.Conv2D(64, 3, padding='same', name='conv_d0a-b')(d0a)
        d0b = layers.ReLU(negative_slope=0.1, name='relu_d0b')(d0b)
        d0c = layers.Conv2D(64, 3, padding='same', name='conv_d0b-c')(d0b)
        d0c = layers.ReLU(negative_slope=0.1, name='relu_d0c')(d0c)

        d1a = layers.MaxPooling2D(pool_size=2, name='pool_d0c-1a')(d0c)
        d1b = layers.Conv2D(128, 3, padding='same', name='conv_d1a-b')(d1a)
        d1b = layers.ReLU(negative_slope=0.1, name='relu_d1b')(d1b)
        d1c = layers.Conv2D(128, 3, padding='same', name='conv_d1b-c')(d1b)
        d1c = layers.ReLU(negative_slope=0.1, name='relu_d1c')(d1c)

        d2a = layers.MaxPooling2D(pool_size=2, name='pool_d1c-2a')(d1c)
        d2b = layers.Conv2D(256, 3, padding='same', name='conv_d2a-b')(d2a)
        d2b = layers.ReLU(negative_slope=0.1, name='relu_d2b')(d2b)
        d2c = layers.Conv2D(256, 3, padding='same', name='conv_d2b-c')(d2b)
        d2c = layers.ReLU(negative_slope=0.1, name='relu_d2c')(d2c)

        d3a = layers.MaxPooling2D(pool_size=2, name='pool_d2c-3a')(d2c)
        d3b = layers.Conv2D(512, 3, padding='same', name='conv_d3a-b')(d3a)
        d3b = layers.ReLU(negative_slope=0.1, name='relu_d3b')(d3b)
        d3c = layers.Conv2D(512, 3, padding='same', name='conv_d3b-c')(d3b)
        d3c = layers.ReLU(negative_slope=0.1, name='relu_d3c')(d3c)
        d3c = layers.Dropout(0.5, name='dropout_d3c')(d3c)

        d4a = layers.MaxPooling2D(pool_size=2, name='pool_d3c-4a')(d3c)
        d4b = layers.Conv2D(1024, 3, padding='same', name='conv_d4a-b')(d4a)
        d4b = layers.ReLU(negative_slope=0.1, name='relu_d4b')(d4b)
        d4c = layers.Conv2D(1024, 3, padding='same', name='conv_d4b-c')(d4b)
        d4c = layers.ReLU(negative_slope=0.1, name='relu_d4c')(d4c)
        d4c = layers.Dropout(0.5, name='dropout_d4c')(d4c)

        u3a = layers.Conv2DTranspose(512, 2, strides=2, name='upconv_d4c_u3a')(d4c)
        u3a = layers.ReLU(negative_slope=0.1)(u3a)
        u3b = layers.Concatenate(name='concat_d3c_u3a-b')([u3a, d3c])
        u3c = layers.Conv2D(512, 3, padding='same', name='conv_u3b-c')(u3b)
        u3c = layers.ReLU(negative_slope=0.1, name='relu_u3c')(u3c)
        u3d = layers.Conv2D(512, 3, padding='same', name='conv_u3c-d')(u3c)
        u3d = layers.ReLU(negative_slope=0.1, name='relu_u3d')(u3d)

        u2a = layers.Conv2DTranspose(256, 2, strides=2, padding='same', name='upconv_u3d_u2a')(u3d)
        u2a = layers.ReLU(negative_slope=0.1)(u2a)
        u2b = layers.Concatenate(name='concat_d2c_u2a-b')([u2a, d2c])
        u2c = layers.Conv2D(256, 3, padding='same', name='conv_u2b-c')(u2b)
        u2c = layers.ReLU(negative_slope=0.1, name='relu_u2c')(u2c)
        u2d = layers.Conv2D(256, 3, padding='same', name='conv_u2c-d')(u2c)
        u2d = layers.ReLU(negative_slope=0.1, name='relu_u2d')(u2d)

        u1a = layers.Conv2DTranspose(128, 2, strides=2, padding='same',name='upconv_u2d_u1a')(u2d)
        u1a = layers.ReLU(negative_slope=0.1)(u1a)
        u1b = layers.Concatenate(name='concat_d1c_u1a-b')([u1a, d1c])
        u1c = layers.Conv2D(128, 3, padding='same', name='conv_u1b-c')(u1b)
        u1c = layers.ReLU(negative_slope=0.1, name='relu_u1c')(u1c)
        u1d = layers.Conv2D(128, 3, padding='same', name='conv_u1c-d')(u1c)
        u1d = layers.ReLU(negative_slope=0.1, name='relu_u1d')(u1d)

        u0a = layers.Conv2DTranspose(128, 2, strides=2, padding='same', name='upconv_u1d_u0a')(u1d)
        u0a = layers.ReLU(negative_slope=0.1, name='relu_u0a')(u0a)
        u0b = layers.Concatenate(name='concat_d0c_u0a-b')([u0a, d0c])
        u0c = layers.Conv2D(128, 3, padding='same', name='conv_u0b-c')(u0b)
        u0c = layers.ReLU(negative_slope=0.1, name='relu_u0c')(u0c)
        u0d = layers.Conv2D(128, 3, padding='same', name='conv_u0c-d')(u0c)
        u0d = layers.ReLU(negative_slope=0.1, name='relu_u0d')(u0d)

        score = layers.Conv2D(2, 1, name='conv_u0d-score')(u0d)
        score = layers.Softmax(name='score')(score)

        return super().__init__(inputs=d0a, outputs=score, name='unet_yeast_bf')

    def load_caffe_weights(self, filename='../weights/2d_cell_net_v0/2d_cell_net_v0.caffemodel.h5'):
        f_weights = h5py.File(filename, 'r')
        for l in self.layers:
            if l.name.split("_")[0] in ["conv", "upconv"]:
                kernel = f_weights['data'][l.name]['0']
                bias = f_weights['data'][l.name]['1']
                kernel = np.transpose(kernel, (2, 3, 1, 0))
                l.set_weights([kernel, bias])
        f_weights.close()

model = UNet()

2026-04-13 23:16:14.214770: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-13 23:16:14.215091: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-13 23:16:14.215256: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [ ]:
# Load Caffe weights
# https://lmb.informatik.uni-freiburg.de/resources/opensource/unet/2d_cell_net_v0_model.zip

model.load_caffe_weights('../weights/2d_cell_net_v0/2d_cell_net_v0.caffemodel.h5')
model.save_weights('../weights/2d_cell_net_v0.hdf5')

In [8]:
def get_training_data():
    im_name_list = [f.rstrip('-BF.tif') for f in sorted(os.listdir('dataset/train')) if f.endswith('-BF.tif')]
    for im_name in im_name_list:
        yield load_image_and_mask('train/'+im_name)
        yield load_image_and_mask('train/'+im_name, simulate_binning=True)

def get_validation_data():
    im_name_list = [f.rstrip('-BF.tif') for f in sorted(os.listdir('dataset/validation')) if f.endswith('-BF.tif')]
    for im_name in im_name_list:
        yield load_image_and_mask('validation/'+im_name)
        yield load_image_and_mask('validation/'+im_name, simulate_binning=True)

def load_image_and_mask(imname, simulate_binning=False):    
    im = imread('dataset/' + imname + '-BF.tif')
    mask = imread('dataset/' + imname + '-Mask.tif')

    if simulate_binning:
        im_binned = (np.array(im[::2, ::2], np.float32) + im[1::2, ::2] + im[::2, 1::2] + im[1::2, 1::2]) / 4
        im_binned[im_binned>65535] = 65535
        im = np.asarray(np.round(im_binned), dtype=np.uint16)

    im_max = im.max()
    im_min = im.min()
    imnorm = np.array(im - im_min, dtype=np.float32) / (im_max - im_min)

    if simulate_binning:
        imnorm = transform.rescale(imnorm, 2)
    
    mask = np.concatenate([1-mask[:, :, np.newaxis], mask[:, :, np.newaxis]], axis=2)

    imnorm = tf.convert_to_tensor(imnorm[np.newaxis, :, :, np.newaxis])
    mask = tf.convert_to_tensor(mask[np.newaxis, :, :, :])

    return (imnorm, mask)

@tf.py_function(Tout=[tf.float32, tf.uint8])
def data_augmentation(im, mask):
    seed = (np.random.randint(0, 0xffff), np.random.randint(0, 0xffff))
    crop_size = 512
    
    im = tf.keras.layers.RandomZoom(height_factor=(-0.2, 0.2), seed=seed[0])(im)
    im = tf.image.stateless_random_crop(im, size=(1, crop_size, crop_size, 1), seed=seed)
    im = tf.image.stateless_random_flip_left_right(im, seed=seed)
    im = tf.image.stateless_random_flip_up_down(im, seed=seed)

    mask = tf.keras.layers.RandomZoom(height_factor=(-0.2, 0.2), seed=seed[0])(mask)
    mask = tf.cast(mask, tf.uint8)
    mask = tf.image.stateless_random_crop(mask, size=(1, crop_size, crop_size, 2), seed=seed)
    mask = tf.image.stateless_random_flip_left_right(mask, seed=seed)
    mask = tf.image.stateless_random_flip_up_down(mask, seed=seed)

    return im, mask

dataset_training = tf.data.Dataset.from_generator(
    get_training_data,
    output_signature=(
        tf.TensorSpec(shape=(1, None, None, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(1, None, None, 2), dtype=tf.uint8),
    )
)
dataset_training = dataset_training.map(data_augmentation)

dataset_validation = tf.data.Dataset.from_generator(
    get_validation_data,
    output_signature=(
        tf.TensorSpec(shape=(1, None, None, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(1, None, None, 2), dtype=tf.uint8),
    )
)

In [ ]:
# ------- Reset Weights
model.load_weights('../weights/2d_cell_net_v0.hdf5')

# ------- Tensorboard Logs
log_dir = "logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    write_images=True
)

# ------- Epoch Checkpoints
checkpoint_dir = log_dir + '/checkpoints'
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_dir + "/epoch{epoch:02d}.weights.h5",
    save_weights_only=True,
    save_best_only=False, 
    save_freq='epoch',
    verbose=True,
)

# ------- Validation Results
file_writer_validation = tf.summary.create_file_writer(log_dir + '/validation')
im_validation_list = [
    load_image_and_mask('validation/'+'20230316-A19-000'),
    load_image_and_mask('validation/'+'20260310-F12-005'),
    load_image_and_mask('train/'+'20260315-F02-001'),
    load_image_and_mask('train/'+'20260315-F02-007'),
    load_image_and_mask('train/'+'20210410-F07-000'),
    load_image_and_mask('validation/'+'20230316-A19-000', simulate_binning=True),
    load_image_and_mask('validation/'+'20260310-F12-005', simulate_binning=True),
]

def validation_train_begin_callback_fn(logs):
    for i, im_validation in enumerate(im_validation_list):
        score = model.predict(im_validation[0])
        with file_writer_validation.as_default():
            tf.summary.image("validation_%d_input" % i, im_validation[0], step=0)
            tf.summary.image("validation_%d_groundtruth" % i, im_validation[1]*255, step=0)
            tf.summary.image("validation_%d_output_before_finetuning" % i, score, step=0)

def validation_epoch_end_callback_fn(epoch, logs):
    for i, im_validation in enumerate(im_validation_list):
        score = model.predict(im_validation[0])
        with file_writer_validation.as_default():
            tf.summary.image("validation_%d_output" % i, score, step=epoch)

validation_callback = tf.keras.callbacks.LambdaCallback(
    on_train_begin=validation_train_begin_callback_fn,
    on_epoch_end=validation_epoch_end_callback_fn,
)

# ------- Training
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossEntropy(from_logits=True),
)

model.fit(dataset_training,
          epochs=10,
          validation_data=dataset_validation,
          callbacks=[tensorboard_callback, validation_callback, checkpoint_callback])

1/1 [==============================] - 0s 23ms/step
Epoch 1/10


2026-04-13 23:28:19.485325: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inunet_yeast_bf/dropout_d3c/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


     36/Unknown - 21s 390ms/step - loss: 0.5387

2026-04-13 23:28:36.907451: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 10638370130832867830
2026-04-13 23:28:36.907511: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 1634035692072526164
2026-04-13 23:28:36.907521: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 1964717055493784344
2026-04-13 23:28:36.907528: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 9003283632777867716
2026-04-13 23:28:36.907553: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 13750974851918342188
2026-04-13 23:28:36.907572: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 16541479075710077852
2026-04-13 23:28:36.907581: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv 

1/1 [==============================] - 0s 23ms/step


2026-04-13 23:28:40.344914: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 1894097824241457406


1/1 [==============================] - 0s 21ms/step

Epoch 1: saving model to logs/20260413-232808/checkpoints/epoch01.weights.h5
36/36 [==============================] - 31s 696ms/step - loss: 0.5387 - val_loss: 0.0974
Epoch 2/10
1/1 [==============================] - 0s 22ms/step

Epoch 2: saving model to logs/20260413-232808/checkpoints/epoch02.weights.h5
36/36 [==============================] - 25s 690ms/step - loss: 0.5347 - val_loss: 0.0980
Epoch 3/10
1/1 [==============================] - 0s 26ms/step

Epoch 3: saving model to logs/20260413-232808/checkpoints/epoch03.weights.h5
36/36 [==============================] - 25s 704ms/step - loss: 0.5281 - val_loss: 0.0897
Epoch 4/10
1/1 [==============================] - 0s 29ms/step

Epoch 4: saving model to logs/20260413-232808/checkpoints/epoch04.weights.h5
36/36 [==============================] - 24s 680ms/step - loss: 0.5265 - val_loss: 0.0990
Epoch 5/10
1/1 [==============================] - 0s 24ms/step

Epoch 5: saving model t

In [ ]:
model.load_weights('logs/20260413-232808/checkpoints/epoch09.weights.h5')
model.save_weights('../unet_yeast_bf/weights/unet_yeast_bf.hdf5')

In [ ]:
# ------ Save as SavedModel
# Load with tf.keras.models.load_model() or tf.saved_model.load()

# Signatures for Tensorflow Serving and tf.saved_model.load()
@tf.function(input_signature=[tf.TensorSpec([None, None, None, 1], tf.float32)])
def serving_fn(input):
    return {'score': model(input)}

model.save("../unet_yeast_bf/weights/unet_yeast_bf/1",
           save_format='tf',
           save_traces=False,
           signatures={'serving_default': serving_fn})

INFO:tensorflow:Assets written to: models/unet_yeast_bf/1/assets


INFO:tensorflow:Assets written to: models/unet_yeast_bf/1/assets


In [ ]:
# ------ Save float16 precision model

tf.keras.backend.set_floatx('float16')

model = UNet()
model.load_weights('logs/20260413-232808/checkpoints/epoch09.weights.h5')
model.save_weights('../unet_yeast_bf/weights/unet_yeast_bf_float16.hdf5')

@tf.function(input_signature=[tf.TensorSpec([None, None, None, 1], tf.float16)])
def serving_fn(input):
    return {'score': model(input)}

model.save("../unet_yeast_bf/weights/unet_yeast_bf_float16/1",
           save_format='tf',
           save_traces=False,
           signatures={'serving_default': serving_fn})

INFO:tensorflow:Assets written to: models/unet_yeast_bf_float16/1/assets


INFO:tensorflow:Assets written to: models/unet_yeast_bf_float16/1/assets
